# ML-04 â€” Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samarthjoshi02/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*   **Unit of analysis**: One row represents a single pseudonymized content item on a specific report date for a specific client (`report_date` × `client_hash_id` × `content_hash_id`).
*   **Time window**: Daily snapshots over a ~17 month panel (2025-01-27 to 2026-06-30), partitioned by month.
*   **What will be predicted**: Whether the page is declining (`is_declining_label`).
*   **What is deliberately excluded**: `trend_pct` and `impressions_last30` because they overlap with the future outcome we want to predict.


In [3]:
import duckdb
import getpass
import os

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    print("Please enter your Hugging Face READ token:")
    hf_token = getpass.getpass()

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("Connection established and authenticated.")


Connection established and authenticated.


## 2. Fields: feature / label / context / excluded

*   **Features**: `impressions_prev30`, `clicks_prev30`, `avg_position_prev30`, `impressions_90d`, `clicks_90d`
*   **Label**: `is_declining_label`
*   **Context**: `client_hash_id`, `content_hash_id`, `report_date`
*   **Excluded**: 
    *   `trend_pct`: Direct target leakage (mathematically derives the label).
    *   `impressions_last30` & `clicks_last30`: Overlap with the label's outcome window (the most recent 30 days).


## 3. Verify it with queries (grain, counts, missing values, windows)


In [6]:
# 1. Grain
q_grain = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
GROUP BY 1, 2, 3
HAVING c > 1
LIMIT 5
"""
print("1. Grain Check (Expect empty dataframe if unique):")
print(conn.sql(q_grain).df())

# 2. Row count + date span
q_span = """
SELECT 
    COUNT(*) as total_rows, 
    MIN(report_date) as start_date, 
    MAX(report_date) as end_date 
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
"""
print("\n2. Row count & Date span:")
print(conn.sql(q_span).df())

# 3. Availability using IS TRUE
q_avail = """
SELECT 
    COUNT(*) as total,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_true,
    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) as ga4_false,
    SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) as ga4_null
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
"""
print("\n3. Availability columns:")
print(conn.sql(q_avail).df())


1. Grain Check (Expect empty dataframe if unique):
  report_date           client_hash_id           content_hash_id  c
0  2026-06-18  client_810019792c9b8efc  content_0064be1867cf7ae8  2
1  2026-06-15  client_a2eeb8899886adde  content_8f1252464688f1c8  2
2  2026-06-15  client_1a8bf67cad4ee525  content_402d0580ddf2e4a6  2
3  2026-06-19  client_1a730cb2640a1abf  content_0ffd552a735a66ab  2
4  2026-06-19  client_a22068e339bf95f5  content_c5dbfd8c5fc4fdb2  2
2. Row count & Date span:
   total_rows start_date   end_date
0    11694072 2026-06-01 2026-06-30
3. Availability columns:
      total  ga4_true  ga4_false   ga4_null
0  11694072  644726.0  8651918.0  2397428.0


## 4. Build the five-feature dataframe & Leakage Experiment

Features:
*   `impressions_prev30`: Knowable at prediction time because it measures traffic exactly before the outcome window.
*   `clicks_prev30`: Knowable at prediction time for the same reason.
*   `avg_position_prev30`: Knowable at prediction time (historical average rank).
*   `impressions_90d`: Knowable at prediction time (full trailing context).
*   `clicks_90d`: Knowable at prediction time (full trailing context).


In [8]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load a sample from the 90-day query table which contains the 30-day aggregations
q_data = """
SELECT impressions_prev30, clicks_prev30, avg_position_prev30, impressions_90d, clicks_90d,
       impressions_last30
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
WHERE impressions_prev30 > 0
LIMIT 50000
"""
df = conn.sql(q_data).df()

# Derive the label and the leaked feature as defined in the data dictionary
df['trend_pct'] = (df['impressions_last30'] - df['impressions_prev30']) / df['impressions_prev30'] * 100
df['is_declining_label'] = (df['trend_pct'] < -20).astype(int)
df = df.fillna(0)

safe_features = ['impressions_prev30', 'clicks_prev30', 'avg_position_prev30', 'impressions_90d', 'clicks_90d']
leaked_feature = 'trend_pct'

# Train WITH LEAKAGE
X_leak = df[safe_features + [leaked_feature]]
y = df['is_declining_label']
X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)

model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train, y_train)
score_leak = roc_auc_score(y_test, model_leak.predict_proba(X_test)[:, 1])
print(f"ROC AUC (WITH LEAKAGE): {score_leak:.4f}")
print("Reason: 'trend_pct' leaks the label perfectly because the label is derived directly from it.\n")

# Train WITHOUT LEAKAGE
X_safe = df[safe_features]
X_train_safe, X_test_safe, _, _ = train_test_split(X_safe, y, test_size=0.2, random_state=42)

model_safe = LogisticRegression(max_iter=1000)
model_safe.fit(X_train_safe, y_train)
score_safe = roc_auc_score(y_test, model_safe.predict_proba(X_test_safe)[:, 1])
print(f"ROC AUC (WITHOUT LEAKAGE): {score_safe:.4f}")
print("Reason: After removing the leaked feature, the model is forced to generalize based on safe historical indicators.")


ROC AUC (WITH LEAKAGE): 1.0000
Reason: 'trend_pct' leaks the label perfectly because the label is derived directly from it.
ROC AUC (WITHOUT LEAKAGE): 0.7143
Reason: After removing the leaked feature, the model is forced to generalize based on safe historical indicators.


## 5. Data limits

*   **Honest limitation**: The panel has highly unbalanced histories per client. Some clients have ~17 months of data, while others have barely 3 months. Generating fixed global windows ignores these entry gaps. We must rely on the client's data start date to safely bound calculations for each client independently.


In [10]:
# Verification complete.
